# Bringing Sketches to Life: my web app

**Author:** Nosa Peter Inwe
**Course:** Computer Vision and Machine Learning II, Université Paris-Est Créteil (UPEC)
**Professor:** Arezoo Ghodsifard

This notebook starts my Gradio web app on a free Colab T4. I can draw a sketch straight in the browser (or upload a photo of one), type a prompt, set the ControlNet strength and generate. It uses the same models, settings and preprocessing as my experiments in `01_sketch_to_image.ipynb`.

Before running: `Runtime → Change runtime type → T4 GPU`. Then run the four cells in order. The last one prints a public `https://….gradio.live` link that stays up for 72 hours at most, or until I stop the cell.

In [ ]:
# 1. I check the GPU. Without one, generating takes minutes per image instead of seconds.
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "No GPU found: switch the runtime to T4 GPU.")

In [ ]:
# 2. I pull my project from GitHub (it's public, so no sign-in needed).
import os
REPO = "/content/gen-ai-projects"
if not os.path.isdir(REPO):
    !git clone -q https://github.com/nosainwe/gen-ai-projects.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}/sketch-to-image-controlnet

In [ ]:
# 3. I install the diffusion libraries and Gradio. Colab already has PyTorch.
!pip install -q "diffusers>=0.35.1,<0.40" "transformers>=4.45,<5" "accelerate>=1,<2" "huggingface-hub>=0.25,<2" "safetensors>=0.4,<1" "gradio>=5.20,<6"

In [ ]:
# 4. I start the app. The models download on my first Generate click (a few GB, a few minutes),
#    then each image takes seconds. I stop this cell to shut the app down.
import sys
sys.path.insert(0, os.getcwd())
from src.gradio_app import SketchApp, build_demo
from src.run_experiments import load_settings

app = SketchApp(load_settings("experiments.json"))
demo = build_demo(app)
demo.queue(max_size=8).launch(share=True, debug=True)